# BTS Flight Departures Scraper (2024)
Scrapes all U.S. airport departure data for 2024 from the BTS On-Time Departures page.

## 1. Imports & Setup

In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
from io import StringIO
import os
import glob

URL = "https://www.transtats.bts.gov/ontime/Departures.aspx"

## 2. Launch Browser & Get All Airports

In [18]:
options = webdriver.ChromeOptions()
DOWNLOAD_DIR = "/Users/lolo/Desktop/Classes/winter'26/PIC16B/Projects/Flight-Delays-Prediction-Model"
options.add_experimental_option("prefs", {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
})

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 15)
driver.get(URL)

## 3. Scrape All Airports (2024)

In [20]:
active_airlines = ['a','b','Delta Airlines Inc. (DL)']
all_data = []
DOWNLOAD_DIR = "/Users/lolo/Desktop/Classes/winter'26/PIC16B/Projects/Flight-Delays-Prediction-Model"

for airport in ["Los Angeles, CA: Los Angeles International (LAX)"]:
    for airline in active_airlines[2:3]:
        print(f"Scraping: {airport} | {airline}")
        try:
            driver.get(URL)
            time.sleep(2)

            airport_dd = Select(wait.until(EC.presence_of_element_located((By.NAME, "cboAirport"))))
            airport_dd.select_by_visible_text(airport)
            time.sleep(1)

            airline_dd = Select(driver.find_element(By.NAME, "cboAirline"))
            airline_dd.select_by_visible_text(airline)

            all_stats_cb = driver.find_element(By.NAME, "chkAllStatistics")
            if not all_stats_cb.is_selected():
                all_stats_cb.click()

            all_months_cb = driver.find_element(By.NAME, "chkAllMonths")
            if not all_months_cb.is_selected():
                all_months_cb.click()

            all_days_cb = driver.find_element(By.NAME, "chkAllDays")
            if not all_days_cb.is_selected():
                all_days_cb.click()

            all_years_cb = driver.find_element(By.NAME, "chkAllYears")
            if all_years_cb.is_selected():
                all_years_cb.click()
            year_cb = driver.find_element(By.XPATH, "//input[@name='chkYears$37']")
            if not year_cb.is_selected():
                year_cb.click()

            driver.find_element(By.XPATH, "//input[@value='Submit']").click()
            time.sleep(4)

            try:
                csv_link = driver.find_element(By.XPATH, "//a[contains(text(), 'CSV')]")
            except:
                print(f"  No data, skipping")
                continue

            for f in glob.glob(os.path.join(DOWNLOAD_DIR, "*.csv")):
                os.remove(f)

            csv_link.click()

            downloaded_file = None
            for _ in range(30):
                time.sleep(1)
                csv_files = glob.glob(os.path.join(DOWNLOAD_DIR, "*.csv"))
                if csv_files:
                    downloaded_file = csv_files[0]
                    break

            if not downloaded_file:
                print(f"  Download timed out, skipping")
                continue

            df = pd.read_csv(downloaded_file)
            df["airport"] = airport
            df["airline"] = airline
            all_data.append(df)
            print(f"  Got {len(df)} rows")

        except Exception as e:
            try:
                Alert(driver).accept()
            except:
                pass
            print(f"  Error: {e}")
            continue

        if len(all_data) > 0 and len(all_data) % 20 == 0:
            pd.concat(all_data, ignore_index=True).to_csv("departures_2024_partial.csv", index=False)
            print("  Progress saved!")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv("departures_2024_all.csv", index=False)
    print(f"Done! Total rows: {len(final_df)}")
else:
    print("No data collected.")

Scraping: Los Angeles, CA: Los Angeles International (LAX) | Delta Airlines Inc. (DL)
  Error: Error tokenizing data. C error: Expected 2 fields in line 4, saw 12

No data collected.


## 4. Save Final CSV & Close Browser

In [22]:
driver.quit()
final_df = pd.read_csv("Detailed_Statistics_Departures.csv", skiprows=7)
final_df

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,DL,01/01/2024,311.0,N513DA,KOA,16:04,15:57,355.0,350.0,-7.0,16:14,17.0,0.0,0.0,0.0,0.0,0.0
1,DL,01/01/2024,318.0,N398DN,DTW,05:30,05:25,271.0,271.0,-5.0,05:46,21.0,0.0,0.0,0.0,0.0,0.0
2,DL,01/01/2024,332.0,N193DN,JFK,07:35,07:32,320.0,328.0,-3.0,08:02,30.0,0.0,0.0,0.0,0.0,0.0
3,DL,01/01/2024,349.0,N411DX,JFK,21:40,21:54,320.0,322.0,14.0,22:16,22.0,14.0,0.0,2.0,0.0,0.0
4,DL,01/01/2024,351.0,N541DE,LIH,14:25,14:21,384.0,371.0,-4.0,14:36,15.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36340,DL,12/31/2024,2975.0,N363NB,PDX,07:00,06:49,154.0,139.0,-11.0,07:00,11.0,0.0,0.0,0.0,0.0,0.0
36341,DL,12/31/2024,2980.0,N377DE,SEA,07:25,07:25,190.0,172.0,0.0,07:39,14.0,0.0,0.0,0.0,0.0,0.0
36342,DL,12/31/2024,8814.0,N553NW,ATL,13:40,13:38,249.0,230.0,-2.0,13:52,14.0,0.0,0.0,0.0,0.0,0.0
36343,DL,12/31/2024,8815.0,N6715C,ATL,15:15,15:08,249.0,214.0,-7.0,15:18,10.0,0.0,0.0,0.0,0.0,0.0
